# Cross-City Transfer Learning for AQI Forecasting in Data-Scarce Indian Regions

**Team Toxic (23)** · Arjun Srikanth, Ankit Umadi, Jayanth K

Can a model trained on other Indian cities improve a target station's daily AQI forecasts when its fitting labels are scarce? The original proposal compares target-only training and cross-city pretraining followed by fine-tuning against persistence at approximately 24, 48 and 72 hours. The refined question asks separately whether pretraining helps, whether subsequent target adaptation helps, and whether either beats practical simple forecasts. The expanded experiment also tests second-order meta-learning, adaptation schedules, donor pool size and larger networks.

This notebook is the entry point to the finished study. Notebook 01 documents source data and exploratory analysis, notebook 02 fits the compact benchmark, and notebook 03 fits the expanded comparison. The tables below are calculated from the completed runs, with forecasts and fitting records retained for verification.


In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reproduce.py').exists())
os.environ.setdefault('AFML_SYNTHESIS_INPUTS', str(ROOT / 'artifacts/AFML_Complete_Project'))
os.environ.setdefault('AFML_SYNTHESIS_OUTPUT', str(ROOT / 'runs/synthesis'))


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

input_setting = os.environ.get("AFML_SYNTHESIS_INPUTS", "/kaggle/input")
input_roots = [Path(path) for path in input_setting.split(os.pathsep)]
output = Path(os.environ.get("AFML_SYNTHESIS_OUTPUT", "/kaggle/working/final_synthesis"))
output.mkdir(parents=True, exist_ok=True)

def find_study_runs(roots):
    found = {}
    for parent in roots:
        for candidate in parent.rglob("frozen_protocol.json"):
            run = candidate.parent
            metadata = run / "models/engine_metadata.json"
            if metadata.exists() and (run / "verification.json").exists():
                count = len(json.loads(metadata.read_text())["methods"])
                assert count not in found, f"More than one audited {count}-method run"
                found[count] = run
    return found

studies = find_study_runs(input_roots)
if set(studies) != {23, 33}:
    archives = [path for root in input_roots for path in root.rglob("synthesis_artifacts.zip")]
    if len(archives) == 1:
        extracted = output / "attached_artifacts"
        extracted.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                assert (extracted / member.filename).resolve().is_relative_to(extracted.resolve())
                assert not member.is_dir() or member.file_size == 0
            archive.extractall(extracted)
        studies = find_study_runs([extracted])
assert set(studies) == {23, 33}, "Attach the audited AFML AQI Research Results dataset"

def artifact(run, relative_name):
    direct = run / relative_name
    if direct.is_file():
        return direct
    if relative_name.endswith(".csv.gz"):
        preserved = run / (relative_name + ".bin")
        if preserved.is_file():
            return preserved
    raise FileNotFoundError(direct)

def read_table(run, relative_name, **kwargs):
    compression = "gzip" if relative_name.endswith(".gz") else None
    return pd.read_csv(artifact(run, relative_name), compression=compression, **kwargs)

def read_record(run, name):
    return json.loads(artifact(run, name).read_text())

def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def audit_checks(run):
    audit = read_record(run, "verification.json")
    assert audit["verdict"] == "PASS" and not audit["critical_failures"]
    paths = {entry["path"]: entry["sha256"] for entry in audit["artifacts"]}
    for name in ("frozen_protocol.json", "models/engine_metadata.json", "models/predictions.csv.gz",
                 "evaluation/macro_metrics.csv", "data/forecast_examples.csv.gz"):
        expected = paths[str(Path(audit["root"]) / name)]
        assert digest(artifact(run, name)) == expected, f"Independent audit hash differs for {name}"
    return len(audit["checks"])

compact, expanded = studies[23], studies[33]
study_records = []
for count, run in studies.items():
    engine = read_record(run, "models/engine_metadata.json")
    state = read_record(run, "models/progress.json")
    assert state["stage"] == "complete"
    study_records.append({"study": "compact" if count == 23 else "expanded", "methods": count,
                          "training_device": engine["device"], "hardware": engine.get("gpu"),
                          "forecasts": state["prediction_rows"], "fit_records": state["fit_rows"],
                          "independent_checks_passed": audit_checks(run)})
study_table = pd.DataFrame(study_records).sort_values("methods")
display(study_table)

main = read_table(expanded, "evaluation/macro_metrics.csv")
main = main.loc[main.period.eq("test_2025") & main.budget.eq(90)]
overview = main.loc[main.groupby("horizon").mae.idxmin(), ["horizon", "method", "mae", "persistence_mae", "skill_mae"]]
display(Markdown("**The 2025 main test at 90 fitting labels.** These are the lowest observed errors at each horizon."))
display(overview.sort_values("horizon").round(3))


## Source, cohort and fitting-label budgets

The archive contains 3,820 original daily AQI workbooks and 769,129 decoded observations. The acquisition audit retains empty exports and unmatched station IDs. Four 2017 observations above 500 remain visible in the quality audit. The modeling years begin in 2020. A missing date stays missing. The extraction discards spreadsheet summary rows and preserves daily values in their original units.

Coverage of at least 80% in each year from 2020 through 2025 leaves 60 stations in 30 cities. For each city, the representative is selected by 2020–2023 observations with station ID breaking ties. Six cities form the target cohort. Every station in those cities is excluded from donor fitting and selection. The 25 km buffer also removes Vatva, near Ahmedabad, leaving 23 donor cities. The rule uses availability through 2025, which favors stations with complete reporting across the study years. The forecast represents one monitor in each city.

Every forecast uses seven consecutive observed AQIs ending on its issue date, plus calendar and horizon features. All horizons use common eligible label dates. A window may require several earlier observations even when a target training budget contains only 30 outcomes. The experiment simulates scarcity of **labels used for fitting** while allowing recent observed context. The budgets contain the latest 30, 90 or 180 eligible labels through 2023. Each date yields three supervised pairs, one for each horizon. Moving to a larger budget changes the dates and seasons represented. Source measurements follow the nominal 4 PM daily AQI convention. Actual publication latency would require separate verification.


In [ ]:
source = read_record(compact, "data/data_summary.json")
assert source["workbooks"] == 3820 and source["observations"] == 769129
display(pd.Series(source, name="Official CPCB release"))
display(read_table(compact, "data/range_flags.csv"))
display(read_table(compact, "data/unmatched_ids.csv"))
cohort = read_table(compact, "data/cohort.csv", dtype={"site_id": str})
display(cohort[["site_id", "station_name", "city", "state", "role", "training_observed_days"]])
eligibility = read_table(compact, "data/eligibility.csv")
display(eligibility[eligibility.period != "adapt"])
for filename in ("eda_source.png", "eda_seasonality.png"):
    display(Image(filename=str(artifact(compact, f"data/{filename}"))))

compact_protocol = read_record(compact, "frozen_protocol.json")
expanded_protocol = read_record(expanded, "frozen_protocol.json")
for name in ("targets", "dev_cities", "budgets", "horizons", "seeds", "latest_fit_cutoff",
             "source_release_sha256", "expected_periods"):
    assert compact_protocol[name] == expanded_protocol[name], name
cutoff = pd.Timestamp(compact_protocol["latest_fit_cutoff"])
date_cols = ["date", "issue_date", "feature_date_min", "feature_date_max"]
examples = read_table(compact, "data/forecast_examples.csv.gz", dtype={"station_id": str}, parse_dates=date_cols)
other_examples = read_table(expanded, "data/forecast_examples.csv.gz", dtype={"station_id": str}, parse_dates=date_cols)
pd.testing.assert_frame_equal(examples, other_examples, check_exact=True)
del other_examples

budget_records = []
for city in compact_protocol["targets"]:
    candidates = examples.loc[examples.city.eq(city) & examples.date.le(cutoff)]
    smaller = set()
    for budget in compact_protocol["budgets"]:
        dates = set(pd.to_datetime(sorted(candidates.date.unique())[-budget:]))
        support = candidates[candidates.date.isin(dates)]
        assert len(dates) == budget and len(support) == 3 * budget and smaller.issubset(dates)
        smaller = dates
        inputs = {day for origin in pd.to_datetime(support.issue_date.unique())
                  for day in pd.date_range(origin - pd.Timedelta(days=6), origin)}
        budget_records.append({"city": city, "budget": budget, "supervised_pairs": len(support),
                               "first_label": min(dates), "last_label": max(dates),
                               "calendar_span_days": (max(dates) - min(dates)).days + 1,
                               "input_dates": len(inputs), "input_dates_outside_label_budget": len(inputs - dates)})
budget_table = pd.DataFrame(budget_records)
display(budget_table)
budget_table.to_csv(output / "fitting_budget_context.csv", index=False)


## Relation to earlier research

RegionTrans motivated transfer across cities with limited observations and demonstrated its method on crowd flow. A location-aware air-quality transfer study used Slovenian PM10 stations and varied the amount of target history. A Saudi AQI study used pollutant and meteorological measurements to forecast an hour ahead. Those tasks differ in target, geography or available inputs. This project contributes a controlled comparison for daily overall AQI from CPCB, with the same observed context for every model, explicit fitting budgets and entire target cities held out. [RegionTrans](https://arxiv.org/abs/1802.00386) · [Cross-station PM10 transfer](https://www.mdpi.com/2079-9292/15/11/2470) · [Hourly AQI forecasting](https://www.mdpi.com/2073-4433/17/9/858)


## Models, temporal validation and meta-learning

Persistence repeats the latest available AQI. Weekly persistence uses the AQI from seven days before the target date. The seven-day mean smooths recent observations. Ridge regression, histogram gradient boosting and a random forest provide three learned local controls. The neural methods use the same inputs and learn a correction to persistence. For issue date $t$ and horizon $h$, they predict $\hat{y}_{t+h}=y_t+\hat{r}_{t,h}$, where $y_t$ is the last observed AQI. For example, a correction of 15 changes an AQI of 100 into a forecast of 115.

The compact benchmark uses five backbone families in local, pooled zero-shot and fine-tuned modes. The expanded experiment uses a stacked GRU, an LSTM with attention pooling, a temporal convolutional network, an encoder Transformer and a residual MLP. It searches hidden widths and training settings on donor data. Zero-shot uses the target's observed recent AQI while fitting on donor labels only. Local models fit only the target budget. Fine-tuned models start from pooled donor weights and update on the same target fitting dates as local models.

The first source fits use 2020–2022 observations from 19 cities. The 2023 donor queries and adaptation on four additional donor development cities select checkpoint, architecture candidate and adaptation settings. Final fitting uses donor data through 2023. Target fitting also ends in 2023. Forecast origins are later than that cutoff. The 2024 target results provide a secondary test of the fixed models. The main test uses 2025 and the same fixed weights, together with rolling observed AQI features.

The expanded meta-learning comparison uses a shared two-layer GRU and stochastic gradient descent for primary target adaptation. Each donor episode uses earlier support labels to adapt and later query labels to assess the update. There are at least nine intervening calendar days. Full MAML differentiates the query loss through the support updates. For one inner step, $\theta'=\theta-\alpha\nabla_\theta L_{support}(\theta)$ and the outer gradient is $(I-\alpha H_{support})^\top\nabla_{\theta'}L_{query}(\theta')$. FOMAML uses the final query gradient with the Hessian term omitted. Reptile interpolates between the starting and adapted parameters. The higher-order gradient check tests that full and first-order methods use the same adapted forward prediction and yield different gradients. [MAML](https://proceedings.mlr.press/v70/finn17a.html) · [Reptile and FOMAML](https://arxiv.org/abs/1803.02999)

The matched ordinary-transfer GRU also uses SGD. Each meta algorithm selects its own settings from the same donor candidate grid, so the test compares the tuned methods as a whole. The Adam sensitivity comparison reuses the corresponding meta initialization and changes the adaptation optimizer and schedule. The smaller donor experiment uses six reproducibly selected cities while keeping the full-pool settings and update counts. The smaller pool changes the number and mix of cities and the volume of unique donor observations. The compact versus expanded comparison also combines changes in capacity, tuning and training schedules. [Local and pooled forecasting](https://arxiv.org/abs/2008.00444)


## Independent recomputation from predictions

MAE measures the average size of forecast errors. RMSE penalizes large errors more heavily. We average absolute and squared errors over dates and seeds within each city, compute the square root for city RMSE, then give each city equal weight. We also examine paired differences at the same dates and use 1,000 moving-block resamples of 14 calendar days within each city. Negative MAE differences favor the named method. An interval describes temporal uncertainty conditional on this cohort. The intervals do not adjust for the many comparisons. The three seeds measure optimization variation for the same stations.


In [ ]:
metric_keys = ["period", "budget", "horizon", "method"]
context_keys = ["period", "city", "station_id", "date", "issue_date", "horizon", "y_true", "last"]
truth_contexts = {}
recomputed = []
change_diagnostics = []
diagnostic_methods = ["persistence", "matched_gru_pooled_zero", "matched_gru_finetune",
                      "maml_gru_sgd", "fomaml_gru_sgd", "reptile_gru_sgd", "deep_transformer_pooled_zero"]
for methods_count, run in sorted(studies.items()):
    protocol = read_record(run, "frozen_protocol.json")
    forecasts = read_table(run, "models/predictions.csv.gz", dtype={"station_id": str},
                           parse_dates=["date", "issue_date"])
    assert len(forecasts) == read_record(run, "models/progress.json")["prediction_rows"]
    assert set(forecasts.method) == set(protocol["expected_methods"])
    assert set(forecasts.budget) == set(protocol["budgets"]) and set(forecasts.seed) == set(protocol["seeds"])
    assert forecasts.issue_date.gt(cutoff).all()
    assert (forecasts.date - forecasts.issue_date).dt.days.eq(forecasts.horizon).all()
    assert not forecasts.duplicated(["period", "city", "station_id", "date", "horizon", "budget", "seed", "method"]).any()
    contexts = forecasts[context_keys].drop_duplicates().sort_values(context_keys).reset_index(drop=True)
    assert not contexts.duplicated(context_keys[:6]).any()
    pairs_per_context = methods_count * len(protocol["budgets"]) * len(protocol["seeds"])
    counts = forecasts.groupby(context_keys, observed=True).size()
    assert counts.eq(pairs_per_context).all()
    truth_contexts[methods_count] = contexts
    error = forecasts.y_pred - forecasts.y_true
    assert np.isfinite(error).all()
    forecasts["absolute_error"] = error.abs()
    forecasts["squared_error"] = error ** 2
    forecasts["signed_error"] = error
    city_scores = forecasts.groupby(metric_keys + ["city"], observed=True).agg(
        mae=("absolute_error", "mean"), mse=("squared_error", "mean"), bias=("signed_error", "mean")
    ).reset_index()
    city_scores["rmse"] = np.sqrt(city_scores.mse)
    actual = city_scores.groupby(metric_keys, observed=True)[["mae", "rmse", "bias"]].mean().reset_index()
    expected = read_table(run, "evaluation/macro_metrics.csv")[metric_keys + ["mae", "rmse", "bias"]]
    compared = actual.merge(expected, on=metric_keys, how="outer", validate="one_to_one", indicator=True,
                            suffixes=("_recalculated", "_saved"))
    assert compared["_merge"].eq("both").all()
    maximum_error = max(float((compared[f"{name}_recalculated"] - compared[f"{name}_saved"]).abs().max())
                        for name in ("mae", "rmse", "bias"))
    assert maximum_error <= 1e-7
    actual["study"] = "compact" if methods_count == 23 else "expanded"
    recomputed.append(actual)
    # Use one budget and average within city before taking the cohort mean.
    changes = forecasts.loc[forecasts.period.eq("test_2025") & forecasts.budget.eq(90),
                           ["city", "method", "horizon", "y_true", "y_pred", "last", "absolute_error"]].copy()
    changes["observed_change"] = changes.y_true - changes["last"]
    changes["forecast_correction"] = changes.y_pred - changes["last"]
    changes["correction_size"] = changes.forecast_correction.abs()
    changes["signed_error"] = changes.y_pred - changes.y_true
    changes["truth_slice"] = np.select([changes.observed_change >= 50, changes.observed_change <= -50],
                                        ["AQI rises by at least 50", "AQI falls by at least 50"],
                                        default="AQI change below 50")
    city_change = changes.groupby(["city", "method", "horizon", "truth_slice"], observed=True).agg(
        observations=("y_true", "size"), mean_correction_size=("correction_size", "mean"),
        mean_correction=("forecast_correction", "mean"), bias=("signed_error", "mean"),
        mean_observed_change=("observed_change", "mean"), mae=("absolute_error", "mean")).reset_index()
    by_change = city_change.groupby(["method", "horizon", "truth_slice"], observed=True).agg(
        cities=("city", "nunique"), seed_forecasts=("observations", "sum"),
        mean_correction_size=("mean_correction_size", "mean"), mean_correction=("mean_correction", "mean"),
        mean_observed_change=("mean_observed_change", "mean"), bias=("bias", "mean"), mae=("mae", "mean")).reset_index()
    by_change["study"] = actual.study.iloc[0]
    by_change["city_dates"] = by_change.pop("seed_forecasts") // len(protocol["seeds"])
    change_diagnostics.append(by_change)
    if methods_count == 33:
        held = forecasts.loc[forecasts.period.eq("test_2025") & forecasts.budget.eq(90)
                             & forecasts.method.isin(diagnostic_methods)]
        daily_diagnostic_errors = held.groupby(["horizon", "city", "date", "method"], observed=True).absolute_error.mean().unstack()
        del held
    print(f"{methods_count} methods. {len(forecasts):,} forecasts. {len(actual):,} recomputed metric cells. "
          f"Maximum absolute discrepancy {maximum_error:.2e}.")
    del forecasts, counts, error, changes, city_change
pd.testing.assert_frame_equal(truth_contexts[23], truth_contexts[33], check_exact=True)
pd.concat(recomputed, ignore_index=True).to_csv(output / "recomputed_macro_metrics.csv", index=False)
change_diagnostics = pd.concat(change_diagnostics, ignore_index=True)
change_diagnostics.to_csv(output / "forecast_change_diagnostics.csv", index=False)


## Answers to the original and refined questions

The original success criterion favors fine-tuning when it improves on the matching local fit and both beat persistence. We evaluate that criterion for MAE and RMSE at each budget and horizon. The refined questions are useful when that conjunction fails. A pooled model can beat persistence even if the later fine-tuning step hurts it, and a fine-tuned model can beat a local model while still losing to persistence. Every comparison below uses the same forecast dates and truthful observed AQI.

| Question | Evidence in this notebook |
|---|---|
| Does pooled training improve on local fitting? | Paired pooled-minus-local errors for each backbone, budget and horizon |
| Does target fitting help a pretrained model? | Paired fine-tuned-minus-pooled errors and the joint proposal criterion |
| Can either beat inexpensive forecasts? | Persistence and strongest observed simple baseline by budget and horizon |
| Does a meta-learned initialization improve transfer? | Shared GRU comparisons with full MAML, FOMAML, Reptile and ordinary fine-tuning |
| Do capacity, adaptation or source-pool changes help? | Matched methods, paired summaries and measured training costs |
| Where do methods fail and what should be tested next? | City and seed errors, high-AQI slices, large-change bias and further work |


In [ ]:
published = {}
for label, run in (("compact", compact), ("expanded", expanded)):
    data = read_table(run, "evaluation/macro_metrics.csv")
    published[label] = data[data.period.eq("test_2025")].copy()

baselines = ["persistence", "seasonal7", "mean7", "ridge_local", "histgb_local", "randomforest_local"]
headlines = []
for tier, metrics in published.items():
    for (budget, horizon), cases in metrics.groupby(["budget", "horizon"]):
        baseline = cases[cases.method.isin(baselines)].nsmallest(1, "mae").iloc[0]
        persistence = cases[cases.method.eq("persistence")].iloc[0]
        best = cases.nsmallest(1, "mae").iloc[0]
        headlines.append({"study": tier, "budget": budget, "horizon": horizon, "lowest_MAE_method": best.method,
                          "MAE": best.mae, "RMSE": best.rmse, "persistence_MAE": persistence.mae,
                          "best_simple_method": baseline.method, "best_simple_MAE": baseline.mae,
                          "MAE_skill_vs_persistence": best.skill_mae, "cities_improved_vs_persistence": best.cities_improved_vs_persistence})
headline = pd.DataFrame(headlines)
display(headline.round(3))
headline.to_csv(output / "main_test_overview.csv", index=False)

contrasts = []
joint = []
for tier, run in (("compact", compact), ("expanded", expanded)):
    comparisons = read_table(run, "evaluation/paired_comparisons.csv")
    comparisons = comparisons[comparisons.period.eq("test_2025")].copy()
    comparisons["study"] = tier
    contrasts.append(comparisons)
    for (budget, horizon), cases in published[tier].groupby(["budget", "horizon"]):
        indexed = cases.set_index("method")
        persist = indexed.loc["persistence"]
        for method in indexed.index:
            if not method.endswith("_finetune"):
                continue
            family = method.removesuffix("_finetune")
            local_name = f"{family}_local"
            if local_name not in indexed.index:
                continue
            local, tuned = indexed.loc[local_name], indexed.loc[method]
            joint.append({"study": tier, "budget": budget, "horizon": horizon, "model": family,
                          "fine_tuned_better_than_local_MAE": tuned.mae < local.mae,
                          "fine_tuned_better_than_local_RMSE": tuned.rmse < local.rmse,
                          "original_success_MAE": tuned.mae < local.mae < persist.mae,
                          "original_success_RMSE": tuned.rmse < local.rmse < persist.rmse})
joint = pd.DataFrame(joint)
joint["original_success_both_errors"] = joint.original_success_MAE & joint.original_success_RMSE
joint.to_csv(output / "original_proposal_success_conditions.csv", index=False)
display(joint.groupby("study").agg(tests=("model", "size"), MAE_success=("original_success_MAE", "sum"),
                                  RMSE_success=("original_success_RMSE", "sum"),
                                  both_errors_success=("original_success_both_errors", "sum")))
for tier, values in joint.groupby("study"):
    wins = int(values.original_success_both_errors.sum())
    display(Markdown(f"The original combined success criterion held in **{wins} of {len(values)}** "
                     f"backbone, budget and horizon comparisons in the {tier} study."))

all_pairs = pd.concat(contrasts, ignore_index=True)
core_kinds = ["pooled_zero_minus_local", "finetune_minus_local", "finetune_minus_pooled_zero"]
core = all_pairs[all_pairs.kind.isin(core_kinds)].copy()
paired_counts = core.assign(improved=core.delta_mae.lt(0), degraded=core.delta_mae.gt(0),
                            interval_favors_method=core.delta_mae_ci_high.lt(0),
                            interval_favors_reference=core.delta_mae_ci_low.gt(0)).groupby(["study", "kind"]).agg(
    cells=("delta_mae", "size"), improved=("improved", "sum"), degraded=("degraded", "sum"),
    interval_favors_method=("interval_favors_method", "sum"),
    interval_favors_reference=("interval_favors_reference", "sum")
).reset_index()
display(paired_counts)
paired_counts.to_csv(output / "core_transfer_comparisons.csv", index=False)
questions = {"pooled_zero_minus_local": "Pooled prediction improved on the matching local neural model",
             "finetune_minus_local": "Fine-tuning improved on the matching local neural model",
             "finetune_minus_pooled_zero": "Target fine-tuning improved on the matching pooled model"}
for result in paired_counts.itertuples(index=False):
    display(Markdown(f"{questions[result.kind]} in **{result.improved}/{result.cells}** "
                     f"main-test comparisons in the {result.study} study. It increased MAE in {result.degraded} comparisons."))

simple_comparisons = []
for tier, metrics in published.items():
    best_simple = metrics[metrics.method.isin(baselines)].sort_values("mae").drop_duplicates(["budget", "horizon"])
    simple_comparisons.append(metrics[~metrics.method.isin(baselines)].merge(
        best_simple[["budget", "horizon", "method", "mae"]].rename(columns={"method": "best_simple", "mae": "best_simple_mae"}),
        on=["budget", "horizon"], validate="many_to_one").assign(study=tier))
simple_comparisons = pd.concat(simple_comparisons, ignore_index=True)
simple_comparisons["mae_minus_best_simple"] = simple_comparisons.mae - simple_comparisons.best_simple_mae
simple_comparisons.to_csv(output / "comparisons_against_best_simple.csv", index=False)
display(simple_comparisons.assign(beats_best_simple=simple_comparisons.mae_minus_best_simple.lt(0)).groupby(
    ["study", "horizon"], observed=True).beats_best_simple.agg(comparisons="count", wins="sum"))

for row in headline.loc[headline.study.eq("expanded") & headline.budget.eq(90)].itertuples(index=False):
    display(Markdown(f"At **{row.horizon} day(s)** and **90 fitting labels**, `{row.lowest_MAE_method}` had the lowest "
                     f"expanded-study MAE at **{row.MAE:.2f}**, compared with **{row.persistence_MAE:.2f}** for persistence "
                     f"and **{row.best_simple_MAE:.2f}** for the strongest observed simple baseline (`{row.best_simple_method}`). "
                     f"It improved on persistence in {row.cities_improved_vs_persistence} of 6 cities."))


## Capacity, meta-learning and donor-pool experiments

Each comparison uses the exact source-verified label dates. We compare the larger models with their same-family compact counterparts. We then compare full MAML, first-order MAML and Reptile with ordinary transfer on the shared GRU. The top results describe the observed test sample. Choosing a deployment method would need a fresh evaluation after selection.


In [ ]:
compact_scores, expanded_scores = published["compact"], published["expanded"]
deep = expanded_scores[expanded_scores.method.str.startswith("deep_")].copy()
deep["compact_method"] = deep.method.str.removeprefix("deep_")
capacity = deep.merge(compact_scores[["budget", "horizon", "method", "mae", "rmse"]].rename(
    columns={"method": "compact_method", "mae": "compact_mae", "rmse": "compact_rmse"}),
    on=["budget", "horizon", "compact_method"], validate="one_to_one")
assert len(capacity) == 5 * 3 * 3 * 3
capacity["large_minus_compact_mae"] = capacity.mae - capacity.compact_mae
capacity["large_minus_compact_rmse"] = capacity.rmse - capacity.compact_rmse
capacity["mode"] = capacity.method.str.split("_", n=2).str[-1]
capacity["lower_MAE"] = capacity.large_minus_compact_mae.lt(0)
capacity.to_csv(output / "capacity_compute_comparison.csv", index=False)
display(capacity.groupby("mode").agg(comparisons=("method", "size"), lower_MAE=("lower_MAE", "sum"),
                                    mean_MAE_difference=("large_minus_compact_mae", "mean")).round(3))
display(Markdown(f"The larger models reduced MAE in **{int(capacity.lower_MAE.sum())} of {len(capacity)}** paired "
                 "architecture, training-mode, fitting-budget and horizon comparisons with their compact counterparts."))

labels = {"native_meta_minus_matched_transfer": "Meta-learning and ordinary GRU fine-tuning",
          "second_order_minus_first_order": "Full MAML and first-order MAML",
          "maml_minus_reptile": "Full MAML and Reptile",
          "optimizer_adam_minus_native_sgd": "Adam and native SGD adaptation",
          "six_donor_pool_minus_all_donors": "Six and 23 donor cities"}
selected_pairs = all_pairs[all_pairs.kind.isin(labels)].copy()
selected_pairs.to_csv(output / "paired_meta_optimizer_and_donor_pool.csv", index=False)
meta_names = [f"{name}_gru_sgd" for name in ("maml", "fomaml", "reptile")]
meta_vs_simple = simple_comparisons.loc[simple_comparisons.study.eq("expanded") & simple_comparisons.method.isin(meta_names)]
assert len(meta_vs_simple) == 27
display(meta_vs_simple.assign(beats_best_simple=meta_vs_simple.mae_minus_best_simple.lt(0)).groupby(
    "method").agg(comparisons=("method", "size"), wins=("beats_best_simple", "sum")))
for kind, title in labels.items():
    view = selected_pairs[selected_pairs.kind.eq(kind)].copy()
    assert not view.empty, kind
    wins = view.delta_mae.lt(0).sum()
    interval_wins = view.delta_mae_ci_high.lt(0).sum()
    interval_losses = view.delta_mae_ci_low.gt(0).sum()
    display(Markdown(f"**{title}.** The named method improved MAE in {wins}/{len(view)} comparisons. "
                     f"Intervals entirely favored the method in {interval_wins} and the reference in {interval_losses}."))
    display(view[view.budget.eq(90)][["method", "reference", "horizon", "delta_mae", "delta_mae_ci_low",
                                     "delta_mae_ci_high", "cities_improved"]].round(3))

display(pd.Series(read_record(expanded, "models/second_order_gradient_check.json"), name="Selected device gradient check"))
numerical_receipt = expanded / "maml_finite_difference_verification.json"
if numerical_receipt.is_file():
    numerical = json.loads(numerical_receipt.read_text())
    assert numerical["verdict"] == "PASS"
    assert numerical["source_module_sha256"] == expanded_protocol["implementation_sha256"]["research_model_engine.py"]
    display(pd.DataFrame(numerical["checks"]))
display(pd.Series(read_record(expanded, "models/diversity_donors.json"), name="Donor-pool composition"))


## When scores change

Reporting aggregate error alone would hide variation across stations and model seeds. These results include the main 2025 test and the secondary 2024 period. Upper-tail diagnostics use the **observed** target AQI to identify days at or above 200. These are retrospective error slices. An operational alert evaluation would group forecasts by a predicted threshold, examine missed alerts and account for actual issue latency. The seasonal rows describe observed groups within the six cities.

The change diagnostics compare the forecast correction with the realized change from issue-day AQI. An upward change with a small or negative correction contributes downward bias. The 50-point slices use observed outcomes and summarize forecast behavior after the fact.

To see whether a comparison depends on one target city, we recalculate each selected error difference with one city omitted. Model weights stay fixed. The 2024–2025 comparison shows whether the direction stays similar as the models age.

For a sensitivity check on temporal uncertainty, we draw the same calendar blocks for all six cities so shared events retain their dates. We try 7-, 14- and 28-day blocks on a fixed set of 90-label comparisons. Each resample uses the observed dates available within each block. The primary intervals above use separate draws within each city.


In [ ]:
featured = ["persistence", "matched_gru_pooled_zero", "matched_gru_finetune", "maml_gru_sgd",
            "fomaml_gru_sgd", "reptile_gru_sgd"]
case = read_table(expanded, "evaluation/case_metrics.csv")
seed_scores = case.groupby(["period", "budget", "horizon", "method", "seed"], observed=True).mae.mean().reset_index()
seed_spread = seed_scores.groupby(["period", "budget", "horizon", "method"], observed=True).mae.agg(
    seed_min="min", seed_max="max", seed_sd="std").reset_index()
seed_spread.to_csv(output / "seed_macro_mae_spread.csv", index=False)
display(seed_spread[(seed_spread.period.eq("test_2025")) & seed_spread.budget.eq(90) & seed_spread.method.isin(featured)].round(3))
city_errors = read_table(expanded, "evaluation/city_metrics.csv")
city_view = city_errors[city_errors.budget.eq(90) & city_errors.horizon.eq(1) & city_errors.method.isin(featured)]
display(city_view.pivot(index=["period", "city"], columns="method", values="mae").round(2))
subgroup = read_table(expanded, "evaluation/subgroup_metrics.csv")
subgroup_view = subgroup[subgroup.period.eq("test_2025") & subgroup.budget.eq(90) & subgroup.horizon.eq(1) & subgroup.method.isin(featured)]
display(subgroup_view[["method", "subgroup_type", "subgroup", "n_cities", "n_city_dates", "mae", "rmse", "bias"]].round(2))
change_view = change_diagnostics[change_diagnostics.study.eq("expanded") & change_diagnostics.horizon.eq(1)
                                 & change_diagnostics.method.isin(featured) & change_diagnostics.mean_observed_change.abs().ge(50)]
display(change_view[["method", "truth_slice", "cities", "city_dates", "mean_observed_change", "mean_correction", "bias", "mae"]].round(2))

# Recalculate contrasts after omitting one test city at a time. The fitted
# models stay fixed. This is a sensitivity analysis of the six-city MAE.
focus_pairs = [(method, "persistence") for method in featured if method != "persistence"]
focus_pairs += [(method, "matched_gru_finetune") for method in featured if method.endswith("_sgd")]
city_key = ["period", "budget", "horizon", "city"]
sensitivity = []
for method, reference in focus_pairs:
    first = city_errors.loc[city_errors.method.eq(method), city_key + ["mae"]]
    second = city_errors.loc[city_errors.method.eq(reference), city_key + ["mae"]]
    deltas = first.merge(second, on=city_key, validate="one_to_one", suffixes=("_method", "_reference"))
    deltas["difference"] = deltas.mae_method - deltas.mae_reference
    for (period, budget, horizon), group in deltas.groupby(["period", "budget", "horizon"]):
        assert len(group) == 6
        total = group.difference.sum()
        leave_one_out = (total - group.difference) / (len(group) - 1)
        sensitivity.append({"period": period, "budget": budget, "horizon": horizon, "method": method,
                            "reference": reference, "all_city_delta_mae": group.difference.mean(),
                            "lowest_leave_one_city_out": leave_one_out.min(),
                            "highest_leave_one_city_out": leave_one_out.max(),
                            "direction_changes": leave_one_out.min() < 0 < leave_one_out.max()})
sensitivity = pd.DataFrame(sensitivity)
sensitivity.to_csv(output / "leave_one_city_out_metric_sensitivity.csv", index=False)
period_comparison = sensitivity.pivot(index=["budget", "horizon", "method", "reference"],
                                     columns="period", values="all_city_delta_mae").reset_index()
period_comparison["same_direction"] = np.sign(period_comparison.secondary_2024).eq(np.sign(period_comparison.test_2025))
period_comparison.to_csv(output / "paired_2024_2025_comparison.csv", index=False)
display(period_comparison[period_comparison.budget.eq(90)].round(3))
display(sensitivity[sensitivity.period.eq("test_2025") & sensitivity.budget.eq(90) & sensitivity.horizon.eq(1)].round(3))

calendar = pd.date_range("2025-01-01", "2025-12-31", freq="D")
bootstrap_pairs = [("matched_gru_pooled_zero", "persistence"), ("deep_transformer_pooled_zero", "persistence")]
bootstrap_pairs += [(method, "matched_gru_finetune") for method in featured if method.endswith("_sgd")]
bootstrap_sensitivity = []
for block_length in [7, 14, 28]:
    rng = np.random.default_rng(20261114 + block_length)
    starts = rng.integers(0, len(calendar) - block_length + 1,
                         size=(1000, int(np.ceil(len(calendar) / block_length))))
    sampled = (starts[:, :, None] + np.arange(block_length)).reshape(1000, -1)[:, :len(calendar)]
    weights = np.vstack([np.bincount(row, minlength=len(calendar)) for row in sampled])
    for (method, reference) in bootstrap_pairs:
        for horizon in [1, 2, 3]:
            daily = daily_diagnostic_errors.xs(horizon, level="horizon")
            differences = (daily[method] - daily[reference]).unstack("city").reindex(calendar)
            counts = weights @ differences.notna().to_numpy(np.float64)
            assert (counts > 0).all()
            draws = ((weights @ differences.fillna(0).to_numpy()) / counts).mean(axis=1)
            lower, upper = np.quantile(draws, [0.025, 0.975])
            point = differences.mean().mean()
            original = all_pairs.loc[all_pairs.study.eq("expanded") & all_pairs.budget.eq(90)
                                     & all_pairs.horizon.eq(horizon) & all_pairs.method.eq(method)
                                     & all_pairs.reference.eq(reference)]
            assert len(original) == 1 and abs(float(original.delta_mae.iloc[0]) - point) < 1e-7
            bootstrap_sensitivity.append({"method": method, "reference": reference, "horizon": horizon,
                                          "synchronized_block_days": block_length, "delta_mae": point,
                                          "ci_low": lower, "ci_high": upper})
bootstrap_sensitivity = pd.DataFrame(bootstrap_sensitivity)
bootstrap_sensitivity.to_csv(output / "calendar_block_uncertainty_sensitivity.csv", index=False)
display(bootstrap_sensitivity[bootstrap_sensitivity.horizon.eq(1)].round(3))

chart = expanded_scores[expanded_scores.budget.eq(90) & expanded_scores.method.isin(featured)]
figure, ax = plt.subplots(figsize=(8.7, 4.6), layout="constrained")
for method in featured:
    curve = chart[chart.method.eq(method)].sort_values("horizon")
    ax.plot(curve.horizon, curve.mae, "o-", label=method.replace("_", " "))
ax.set(xlabel="Forecast horizon in days", ylabel="Mean city MAE in AQI units", xticks=[1, 2, 3],
       title="2025 forecast errors with 90 target fitting labels")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, fontsize=8, ncol=2)
figure.savefig(output / "matched_meta_and_transfer_horizons.png", dpi=170)
plt.show()
plt.close(figure)
for name in ["learning_curves.png", "per_city_skill.png", "forecast_traces.png"]:
    display(Image(filename=str(artifact(expanded, f"evaluation/{name}"))))


## What the additional computation bought

The expanded search fits four donor candidates per supervised backbone and four per meta-learning method. The final three seeds use the choices from the donor search. A selected checkpoint at the training limit shows that the search ended while that candidate still had its best measured result. Shared donor fitting, candidate selection and per-target adaptation contribute different kinds of cost. The recorded times include fitting and validation work and are specific to the execution hardware. Approximate FLOP proxies are available in the expanded fit records.


In [ ]:
costs = []
for label, run in (("compact", compact), ("expanded", expanded)):
    fits = read_table(run, "models/fit_log.csv")
    fits["study"] = label
    if "optimizer" in fits:
        # The engine uses Reptile parameter interpolation. Correct the source-fit
        # reporting label here while keeping the original journal intact.
        reptile_source = fits.method.eq("reptile") & fits.stage.str.startswith("donor_")
        fits.loc[reptile_source, "optimizer"] = "parameter interpolation"
    costs.append(fits)
costs = pd.concat(costs, ignore_index=True)
costs.to_csv(output / "fit_log_with_reporting_labels.csv", index=False)
by_stage = costs.groupby(["study", "stage"], observed=True).agg(
    fits=("method", "size"), seconds=("fit_seconds", "sum"), maximum_parameters=("parameters", "max")).reset_index()
display(by_stage.round(2))
by_stage.to_csv(output / "cost_by_study_and_stage.csv", index=False)

search = costs[costs.stage.isin(["donor_hpo_source", "donor_hpo_meta"])].copy()
search["selected_at_limit"] = search.selected_step.eq(search.requested_steps)
display(search.groupby(["stage", "method"], observed=True).agg(
    candidate_fits=("method", "size"), total_updates=("actual_steps", "sum"),
    checkpoint_at_limit=("selected_at_limit", "sum"), seconds=("fit_seconds", "sum")).round(1))
display(pd.Series(read_record(expanded, "models/compute_summary.json"), name="Expanded execution"))
display(Markdown("The original Reptile source-fit journal says `outer_adam`. The code interpolates parameters. "
                 "The derived fit log uses the code's actual update rule."))

selected_sources = read_record(expanded, "models/source_selections.json")
specs = read_record(expanded, "models/selected_model_specs.json")
choices = []
for family, selected in selected_sources["supervised"].items():
    choices.append({"model": family, "hidden_width": specs[family]["hidden"], "refit_updates": selected["steps"],
                    "optimizer_lr": selected["preset"]["lr"], "weight_decay": selected["preset"]["weight_decay"]})
display(pd.DataFrame(choices))

benchmarks_90 = expanded_scores[expanded_scores.budget.eq(90) & expanded_scores.horizon.eq(1)].copy()
target_cost = costs[costs.study.eq("expanded") & costs.stage.eq("target_fit") & costs.budget.eq(90)]
target_cost = target_cost.groupby("method", observed=True).agg(parameters=("parameters", "max"),
                                                               median_target_fit_seconds=("fit_seconds", "median")).reset_index()
deployment_table = benchmarks_90[["method", "mae", "rmse"]].merge(target_cost, on="method", validate="one_to_one")
deployment_table.to_csv(output / "main_test_error_and_target_fit_cost_90_labels.csv", index=False)
display(deployment_table.sort_values("mae").head(12).round(4))


## Interpretation and further work

The test cohort supports a controlled comparison of machine learning choices for stations with enough reporting to construct the same input windows. Coverage selected through 2025, six held-out cities and one station per city limit geographic inference. A 30-label fitting budget still assumes seven recent AQIs for each scored origin. The latest-label design changes the season mix and elapsed time with budget. The fixed cutoff also lets the models age before the 2025 test. Seasonal slices and upper-tail errors can motivate explanations for failure. Establishing a cause requires a specific experiment that isolates it.

The study answers whether the methods help under these stated budgets, horizons and dates. To test operation in more data-scarce settings, preselect additional cities and a separate test period, include stations with missing recent observations, and evaluate roll-forward updates with documented data publication times. Match total source observation volume and vary donor geography repeatedly to distinguish diversity from data quantity. Align weather forecasts and peer sensors to the information available at the issue date before testing their contribution. Calibrate high-AQI alerts on new labeled outcomes after choosing an alert policy.

The full results include the audit of original data and labels, 2023 donor selection evidence, both target evaluation years, per-date forecasts, paired uncertainty, seed variation and fitting costs. The CPCB archive and the experiment code specify how to rerun the study. [CPCB archive](https://airquality.cpcb.gov.in/ccr/#/repository/aqi) · [Time-series evaluation](https://link.springer.com/article/10.1007/s10618-022-00894-5)


In [ ]:
receipt = {
    "studies": study_table.to_dict(orient="records"),
    "source_release_sha256": compact_protocol["source_release_sha256"],
    "shared_examples_and_evaluation_contexts_verified": True,
    "independent_artifact_hash_checks_passed": True,
    "saved_macro_mae_rmse_and_bias_recomputed": True,
    "main_test": "2025",
    "secondary_test": "2024",
}
(output / "synthesis_verification.json").write_text(json.dumps(receipt, indent=2))
print("Verified synthesis outputs", sorted(path.name for path in output.iterdir() if path.is_file()))
